<a href="https://colab.research.google.com/github/lambdabypi/AppliedGenAIIE5374/blob/main/M11_Lab2_Multi_Agent_Investment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💹 <span style="color:#2c3e50;">Advanced CrewAI:</span> <span style="color:#16a085;">Multi-Agent Investment Analysis</span> with Delegation & RAG

## 📘 <span style="color:#34495e;">Lab Overview</span>

You already know the basics of CrewAI — now let’s explore its **advanced features** that unlock real-world power.  
In this lab, you'll build a **sophisticated investment analysis system** demonstrating:

### 🎯 <span style="color:#2980b9;">Advanced Features You'll Master</span>
- 🧠 <strong>Agent Delegation</strong> – Let agents automatically assign work to specialists  
- 📄 <strong>RAG Integration</strong> – Analyze uploaded financial documents with AI  
- 📈 <strong>Real-time Data</strong> – Combine live market data with AI analysis  
- 📝 <strong>Professional Output</strong> – Transform messy AI responses into clean reports

### 💼 <span style="color:#8e44ad;">What You're Building</span>
A **4-agent investment team** that works like a real Wall Street firm:  
- The **Portfolio Manager** delegates to specialists  
- The **Research Analyst** reads your uploaded documents  
- The team produces **investment recommendations** using live data

### 🔥 <span style="color:#c0392b;">Why This Matters</span>
These patterns — delegation, RAG, real-time integration — are **essential for building production-grade AI systems** that can handle complex, multi-step workflows across any domain.


In [2]:
# +++++ 📦 Package Installation
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Install required packages for advanced CrewAI features and financial data integration

!pip install -q crewai crewai-tools langchain-openai yfinance plotly qdrant-client
print("✅ All packages installed successfully!")

# 📌 Package Explanations:
# - crewai: Core library to define and manage multi-agent AI workflows.
# - crewai-tools: Adds tools and enhancements to improve agent capabilities in CrewAI.
# - langchain-openai: Enables integration of OpenAI LLMs with LangChain for natural language processing.
# - yfinance: Used to fetch real-time and historical financial data from Yahoo Finance.
# - plotly: Enables creation of interactive and visually appealing charts for data analysis.

# Start time tracking (put this at the beginning of your lab)
import time
from datetime import datetime
start_time = time.time()

✅ All packages installed successfully!


In [3]:
# +++++ 🎨 Pretty Print Utility
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create styled message boxes for better visual output in Colab

from IPython.display import display, HTML

def pretty_print(text, title="ℹ️ Info", theme="blue"):
    """Displays a styled message box with optional color themes: blue, red, or yellow."""

    themes = {
        "blue": {"color": "#1e4b8f", "background": "#f0f6ff"},
        "red": {"color": "#c62828", "background": "#ffebee"},
        "yellow": {"color": "#b26a00", "background": "#fff8e1"}
    }

    style = themes.get(theme.lower(), themes["blue"])
    formatted_text = text.replace('\n', '<br>')

    display(HTML(f"""
    <div style="border-left: 5px solid {style['color']}; padding: 12px 16px; background-color: {style['background']};
                border-radius: 6px; font-family: 'Segoe UI', sans-serif; line-height: 1.6; margin: 10px 0;">
        <strong style="color: {style['color']}; font-size: 16px;">{title}</strong><br>
        <span style="font-size: 14px; color: #333;">{formatted_text}</span>
    </div>
    """))

print("🎨 Pretty print utility ready!")

🎨 Pretty print utility ready!


In [4]:
# +++++ 🔑 API Setup & Authentication
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Secure API key setup from Google Colab secrets

try:
    from google.colab import userdata
    import os
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    pretty_print("🔐 OpenAI API key successfully loaded. You're authenticated and ready to go!", "✅ API Key Setup", "blue")
except:
    pretty_print("⚠️ OpenAI API key not found. Please set it in Colab ➤ More ➤ Secrets before running the lab.", "❌ Missing API Key", "red")


## 🛠️ Advanced Tools & Agent Setup

Now we'll set up the advanced CrewAI components that make this system powerful:

**🔧 Specialized Tools:**
- **Financial Web Scrapers** - Extract live data from Yahoo Finance and MarketWatch
- **RAG File Reader** - Analyze your uploaded financial documents (PDFs, reports)
- **Web Search** - General purpose research capabilities

**👥 The 4-Agent Investment Team:**
1. **📊 Portfolio Manager** - Has delegation powers, coordinates the entire analysis
2. **📰 Market Analyst** - Scrapes financial websites for current news and sentiment
3. **📚 Research Analyst** - Uses RAG to read and analyze your uploaded documents
4. **💹 Trading Strategist** - Synthesizes everything into actionable recommendations

**🔥 Key Advanced Features:**
- **Delegation**: Portfolio Manager can automatically assign tasks to specialists
- **RAG**: Research Analyst reads YOUR uploaded files (earnings reports, SEC filings)
- **Specialization**: Each agent has specific tools and expertise areas

In [5]:
# +++++ 🛠️ Import Libraries & Initialize Tools
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Import CrewAI framework and set up specialized tools for financial analysis

from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
# Built-in tools for web search, web scraping, and file reading
from crewai_tools import WebsiteSearchTool, ScrapeWebsiteTool, FileReadTool

# FINANCIAL DATA SCRAPING TOOLS:
yahoo_finance_scraper = ScrapeWebsiteTool(website_url='https://finance.yahoo.com')    # Live stock prices
marketwatch_scraper = ScrapeWebsiteTool(website_url='https://www.marketwatch.com')    # Market news
web_search = WebsiteSearchTool()                                                       # General web search

# RAG (Retrieval-Augmented Generation) TOOL:
file_reader = FileReadTool()  # Reads PDFs, text files, and documents you upload

# AI MODEL CONFIGURATION:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # Lower temp = more consistent output

pretty_print("Investment tools locked and loaded!", "🔧 Tools Ready", "blue")


In [18]:
# +++++ 👥 Create Specialized AI Agents with Delegation
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create 4 specialized AI agents that work together like a real investment team

# AGENT 1: PORTFOLIO MANAGER (THE BOSS)
portfolio_manager = Agent(
    role="Senior Portfolio Manager",
    goal="Make strategic investment decisions by coordinating research team",
    backstory="You're a seasoned Wall Street portfolio manager with 15+ years experience. You delegate research tasks to specialists and make final investment calls based on comprehensive analysis.",
    llm=llm,
    tools=[web_search],
    allow_delegation=True,  # 🔥 DELEGATION POWER! This agent can delegate tasks to others
    max_delegation=3,       # Can delegate up to 3 tasks at once
    verbose=False
)
# 📌 This agent acts as the team leader, assigning tasks and integrating all outputs to make final investment decisions.
# Think of it as the executive strategist who oversees the whole operation and ensures every agent’s work aligns with the firm’s goals.

# AGENT 2: MARKET NEWS ANALYST
market_analyst = Agent(
    role="Market News Analyst",
    goal="Track real-time market news and sentiment for specific stocks",
    backstory="You're a former financial journalist who now specializes in analyzing market news, earnings reports, and sentiment. You have your finger on the pulse of Wall Street.",
    llm=llm,
    tools=[yahoo_finance_scraper, marketwatch_scraper, web_search],  # Has access to financial websites
    verbose=False
)
# 📰 This agent monitors live financial news and trends from trusted sources to detect any signals or events that may impact stock prices.
# It plays a critical role in sentiment analysis and contextual understanding of market movement.

# AGENT 3: RESEARCH DOCUMENT ANALYST (RAG SPECIALIST)
research_analyst = Agent(
    role="Research Document Analyst",
    goal="Analyze financial documents, earnings reports, and research files",
    backstory="You're a CFA charterholder who excels at digging through financial documents, SEC filings, and research reports to find hidden insights and key metrics.",
    llm=llm,
    tools=[file_reader, web_search],  # Can read your uploaded files + web search
    verbose=False
)
# 📄 This agent is the RAG powerhouse — it reads user-uploaded documents and extracts valuable insights.
# It’s ideal for deep analysis of PDFs, reports, and any offline data that supports investment decisions.

# AGENT 4: TRADING STRATEGIST
trading_strategist = Agent(
    role="Trading Strategist",
    goal="Synthesize all research into actionable BUY/SELL/HOLD recommendations",
    backstory="You're a quantitative analyst who combines technical analysis, fundamental analysis, and market sentiment to create clear, actionable trading strategies with specific price targets.",
    llm=llm,
    tools=[web_search],
    verbose=False
)
# 💹 This is the final decision-maker who converts all research into real trading signals.
# It crafts buy/sell/hold strategies with target prices by balancing risk, trend, and fundamental indicators.

pretty_print("💼 Investment dream team assembled!\n📊 Portfolio Manager (Boss)\n📰 Market Analyst\n📚 Research Analyst\n💹 Trading Strategist", "👥 Team Ready", "blue")


## 📋 Task Definition & Workflow Design

Here's where the advanced CrewAI features really shine. We'll create tasks that demonstrate:

**🎯 Delegation in Action:**
The Portfolio Manager doesn't do the work directly - instead, it **delegates** specific tasks to the right specialists and then **coordinates** their findings into a final recommendation.

**📚 RAG Implementation:**
The Research Analyst can read and analyze any financial documents you upload (earnings reports, SEC filings, research papers) and extract key insights that wouldn't be available through web search alone.

**🔄 Workflow Process:**
1. Portfolio Manager **delegates** news analysis to Market Analyst
2. Portfolio Manager **delegates** document analysis to Research Analyst  
3. Portfolio Manager **delegates** strategy creation to Trading Strategist
4. Portfolio Manager **synthesizes** all findings into executive summary

**💡 Why This Architecture Works:**
Just like a real investment firm, specialization + coordination produces better results than any single agent trying to do everything.

In [19]:
# +++++ 📋 Define Agent Tasks with Specific Formats
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create specific tasks for each agent with exact output formats to avoid messy results

def create_investment_tasks(stock_symbol):
    """
    Creates 4 specialized tasks that demonstrate delegation and RAG capabilities.
    Each task has specific output format to prevent messy results.
    """

    # TASK 1: PORTFOLIO MANAGER COORDINATION TASK (DELEGATION)
    coordination_task = Task(
        description=f"""As Portfolio Manager, coordinate investment analysis for {stock_symbol}.

        DELEGATE to your team and create EXECUTIVE SUMMARY in this EXACT format:

        🏢 {stock_symbol} INVESTMENT ANALYSIS
        💰 RECOMMENDATION: [BUY/SELL/HOLD]
        🎯 Target Price: $[X.XX]
        📉 Stop Loss: $[X.XX]
        ⏰ Timeframe: [Short/Medium/Long term]

        KEY INSIGHTS (3 bullet points max):
        • [Key point 1]
        • [Key point 2]
        • [Key point 3]

        ⚠️ RISKS: [Main risk in 1 sentence]

        Keep it SHORT and ACTIONABLE. No paragraphs!""",
        agent=portfolio_manager,
        expected_output=f"Clean executive summary for {stock_symbol} with exact format"
    )

    # TASK 2: MARKET NEWS ANALYSIS TASK (WEB SCRAPING)
    market_task = Task(
        description=f"""Find latest news for {stock_symbol} and summarize in EXACTLY this format:

        📰 MARKET NEWS ({stock_symbol})
        • [Latest news headline 1]
        • [Latest news headline 2]
        • [Latest news headline 3]

        📊 SENTIMENT: [Positive/Negative/Neutral] - [Why in 1 sentence]

        Keep it SHORT! Max 4 lines total.""",
        agent=market_analyst,
        expected_output=f"Short news summary for {stock_symbol} in exact format"
    )

    # TASK 3: DOCUMENT ANALYSIS TASK (RAG)
    research_task = Task(
        description=f"""Analyze {stock_symbol} financials and provide EXACTLY this format:

        📊 FINANCIAL HEALTH ({stock_symbol})
        • Revenue: [Growing/Declining/Stable]
        • Profit: [Strong/Weak/Average]
        • Debt: [Low/Medium/High]

        💡 KEY METRIC: [Most important number]

        Keep it SHORT! Max 4 lines total.""",
        agent=research_analyst,
        expected_output=f"Short financial summary for {stock_symbol} in exact format"
    )

    # TASK 4: TRADING STRATEGY TASK (SYNTHESIS)
    strategy_task = Task(
        description=f"""Create trading strategy for {stock_symbol} in EXACTLY this format:

        💹 TRADING STRATEGY ({stock_symbol})
        >>  Today's Price ($[X.XX])
        🎯 Action: [BUY/SELL/HOLD]
        💰 Entry Price: $[X.XX]
        🚀 Target: $[X.XX]
        🛑 Stop Loss: $[X.XX]

        📈 WHY: [Reason in 1 sentence]

        Keep it SHORT! Max 5 lines total.""",
        agent=trading_strategist,
        expected_output=f"Short trading strategy for {stock_symbol} in exact format"
    )

    return coordination_task, market_task, research_task, strategy_task

print("📋 Task templates created - ready for delegation!")

📋 Task templates created - ready for delegation!


## 🚀 Crew Assembly & Advanced Output Processing

The final step brings everything together with two key advanced features:

**🎯 Multi-Agent Orchestration:**
The `analyze_stock()` function creates a crew where agents can delegate tasks to each other, work in parallel when possible, and coordinate their findings automatically.

**📊 Real-Time Data Integration:**
The `clean_investment_output()` function demonstrates how to enhance AI analysis with live data - it fetches current stock prices, recent trading history, and key metrics from financial APIs, then formats everything into a professional investment report.

**💡 Why This Approach Works:**
- **Delegation** ensures the right specialist handles each task
- **RAG** incorporates your private documents into the analysis  
- **Real-time data** keeps recommendations current and actionable
- **Clean formatting** transforms messy AI output into professional reports

This pattern can be adapted for any domain where you need specialized AI agents working with both private documents and live data sources.

In [20]:
# +++++ 🚀 Crew Assembly & Execution Function
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create main function that assembles all agents into a working crew

def analyze_stock(stock_symbol):
    """
    Main function that runs the complete investment analysis workflow.
    Demonstrates multi-agent collaboration with delegation and real-time data integration.
    """

    # Create tasks for all 4 agents
    coord_task, market_task, research_task, strategy_task = create_investment_tasks(stock_symbol)

    # Assemble agents into crew - enables delegation between agents
    investment_crew = Crew(
        agents=[portfolio_manager, market_analyst, research_analyst, trading_strategist],
        tasks=[coord_task, market_task, research_task, strategy_task],
        verbose=True,          # Keeps output clean
        max_iter=5,            # Allows for delegation loops
        output_log_file=False  # Prevents messy log files
    )

    pretty_print(f"Analyzing {stock_symbol}...", "💹 Processing", "yellow")

    # Execute crew - this is where delegation happens automatically
    result = investment_crew.kickoff()

    return result

pretty_print("✅ Main analysis logic is ready to run!", title="🟢 CrewAI Initialized")

In [21]:
# +++++ 🎯 Execute Advanced CrewAI Analysis
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


# Execute the complete analysis for Apple stock
print("Running advanced CrewAI analysis with delegation and real-time data...")
result = analyze_stock('AAPL')
print(result)

pretty_print("✅ Analysis complete! Your AI investment team used delegation to coordinate specialists, integrated real-time market data, and produced a professional investment report.", "🎊 Success", "blue")

Running advanced CrewAI analysis with delegation and real-time data...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fc954505-5a57-4703-8c00-8e8d3431f18c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Portfolio Manager                                                                                │
│                                                                                                                 │
│  Task: As Portfolio Manager, coordinate investment analysis for AAPL.                                           │
│                                                                                                                 │
│          DELEGATE to your team and create EXECUTIVE SUMMARY in this EXACT format:                               │
│                                                                                                                 │
│          🏢 AAPL INVESTMENT ANALYSIS                                                                            │
│          💰 RECOMMENDATION: [BUY/SELL/HOLD]                                                                     │
│          🎯 Target Price: $[X.XX]                                                                               │
│          📉 Stop Loss: $[X.XX]                                                                                  │
│          ⏰ Timeframe: [Short/Medium/Long term]                                                                 │
│                                                                                                                 │
│          KEY INSIGHTS (3 bullet points max):                                                                    │
│          • [Key point 1]                                                                                        │
│          • [Key point 2]                                                                                        │
│          • [Key point 3]                                                                                        │
│                                                                                                                 │
│          ⚠️ RISKS: [Main risk in 1 sentence]                                                                     │
│                                                                                                                 │
│          Keep it SHORT and ACTIONABLE. No paragraphs!                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Portfolio Manager                                                                                │
│                                                                                                                 │
│  Thought: I need to gather the necessary information for the investment analysis of AAPL by coordinating with   │
│  my research team.                                                                                              │
│                                                                                                                 │
│  Using Tool: Search in a specific website                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trading Strategist                                                                                      │
│                                                                                                                 │
│  Thought: Action: Search in a specific website                                                                  │
│                                                                                                                 │
│  Using Tool: Search in a specific website                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trading Strategist                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  💹 TRADING STRATEGY (AAPL)                                                                                     │
│          >>  Today's Price ($[X.XX])                                                                            │
│          🎯 Action: BUY                                                                                         │
│          💰 Entry Price: $[X.XX]                                                                                │
│          🚀 Target: $300.00                                                                                     │
│          🛑 Stop Loss: $250.00                                                                                  │
│                                                                                                                 │
│          📈 WHY: Strong brand loyalty and growth in services despite recent legal challenges.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

In [22]:
# +++++ 📊 HTML Investment Report (Dropbox Version with CrewAI Data)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Download HTML report from Dropbox and use it with real CrewAI result data

import requests

print("🔄 Loading HTML investment report from Dropbox...")

# Download the HTML report script
dropbox_url = "https://www.dropbox.com/scl/fi/kwuifm4ullucyvyxnnax0/HTML-Investment-Report.html?rlkey=jkfjmxnoks7rxwuxfvh46slap&dl=1"

response = requests.get(dropbox_url)
code_content = response.text

# Execute the code to load the function
exec(code_content)

print(f"🎨 Generating HTML report with CrewAI result data...")

# Pass the result from your earlier CrewAI analysis
html_report = create_html_investment_report("AAPL", result)

display(HTML(html_report))

print("✅ HTML Investment Report with real AI recommendations displayed!")

🔄 Loading HTML investment report from Dropbox...
🎨 Generating HTML report with CrewAI result data...
ðŸ”� Debug - Portfolio output: 🏢 AAPL INVESTMENT ANALYSIS
💰 RECOMMENDATION: BUY
🎯 Target Price: $300.00
📉 Stop Loss: $250.00
⏰ Timeframe: Medium term

KEY INSIGHTS:
• Strong brand loyalty and market position in consumer electronics...
ðŸ”� Debug - Trading output: 💹 TRADING STRATEGY (AAPL)
        >>  Today's Price ($[X.XX])
        🎯 Action: BUY
        💰 Entry Price: $[X.XX]
        🚀 Target: $300.00
        🛑 Stop Loss: $250.00

        📈 WHY: Strong brand l...
ðŸ“Š Extracted - Rec: BUY, Target: $300.00, Stop: $250.00, Entry: $[X.XX]


✅ HTML Investment Report with real AI recommendations displayed!


## 🧪 Hands-On Lab: Customize and Explore Your Investment Crew

In this lab, you'll modify your CrewAI investment model by experimenting with agent roles, adding financial tools, and testing different stock symbols. Follow the steps below and submit your observations.

---

### <span style="color:#3b82f6; font-weight:bold;">1. Modify the Strategic Agent</span>
Update the Strategic Agent’s role, goal, or tools to see how it affects the model.
- You might make it more risk-focused or give it access to tools like news search or valuation metrics.

---

### <span style="color:#3b82f6; font-weight:bold;">2. Add a Financial Tool or Calculator</span>
Create a simple helper function or add a tool to compute key financial metrics.
- For example: risk/reward ratio, moving averages, or P/E ratio.

---

### <span style="color:#3b82f6; font-weight:bold;">3. Test New Stock Symbols</span>
Run the model using at least three other stock symbols:
- Suggestions: `TSLA`, `GOOGL`, and `NVDA`. Compare how the recommendations change across different companies.

---

### <span style="color:#3b82f6; font-weight:bold;">4. Submit a 1-Page PDF Report</span>
Write a brief summary covering:What you changed, What symbols you tested, What you observed

Export the report as a **1-2 page PDF**.

---

### <span style="color:#3b82f6; font-weight:bold;">5. Confirm Your Submission</span>
Complete the next cell to finalize your lab submission.
- Make sure your code is saved and your PDF is ready.

---


In [11]:
# +++++ 🧪 LAB SECTION 1: Modify the Strategic Agent
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Update the Trading Strategist to be more risk-focused with enhanced capabilities

# MODIFIED AGENT 4: RISK-FOCUSED TRADING STRATEGIST
trading_strategist_v2 = Agent(
    role="Risk-Focused Trading Strategist",
    goal="Create conservative trading strategies with emphasis on risk management and downside protection",
    backstory="""You're a quantitative risk analyst with expertise in portfolio protection strategies.
    You prioritize capital preservation and use strict risk/reward ratios (minimum 2:1).
    You incorporate volatility analysis, technical indicators, and fundamental valuation metrics
    to create defensive trading strategies with clear exit plans.""",
    llm=llm,
    tools=[web_search],
    verbose=False
)

pretty_print(
    "✅ Trading Strategist upgraded to Risk-Focused version!\n"
    "New features:\n"
    "• Emphasis on capital preservation\n"
    "• Minimum 2:1 risk/reward ratios\n"
    "• Volatility and valuation analysis\n"
    "• Conservative position sizing",
    "🔄 Agent Modified",
    "blue"
)

In [12]:
# +++++ 🧪 LAB SECTION 2: Add Financial Calculation Tools
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create helper functions for key financial metrics

import yfinance as yf
import numpy as np
from datetime import datetime, timedelta

def calculate_risk_reward_ratio(entry_price, target_price, stop_loss):
    """Calculate risk/reward ratio for a trade"""
    potential_profit = target_price - entry_price
    potential_loss = entry_price - stop_loss

    if potential_loss <= 0:
        return 0

    ratio = potential_profit / potential_loss
    return round(ratio, 2)

def get_financial_metrics(stock_symbol):
    """Fetch key financial metrics using yfinance"""
    try:
        ticker = yf.Ticker(stock_symbol)
        info = ticker.info

        metrics = {
            'current_price': info.get('currentPrice', 'N/A'),
            'pe_ratio': info.get('trailingPE', 'N/A'),
            'forward_pe': info.get('forwardPE', 'N/A'),
            'peg_ratio': info.get('pegRatio', 'N/A'),
            'price_to_book': info.get('priceToBook', 'N/A'),
            'dividend_yield': info.get('dividendYield', 0) * 100 if info.get('dividendYield') else 'N/A',
            'market_cap': info.get('marketCap', 'N/A'),
            '52week_high': info.get('fiftyTwoWeekHigh', 'N/A'),
            '52week_low': info.get('fiftyTwoWeekLow', 'N/A'),
            'beta': info.get('beta', 'N/A')
        }

        return metrics
    except Exception as e:
        return {'error': str(e)}

def calculate_moving_averages(stock_symbol, periods=[20, 50, 200]):
    """Calculate simple moving averages"""
    try:
        ticker = yf.Ticker(stock_symbol)
        hist = ticker.history(period="1y")

        ma_data = {}
        for period in periods:
            if len(hist) >= period:
                ma = hist['Close'].rolling(window=period).mean().iloc[-1]
                ma_data[f'MA_{period}'] = round(ma, 2)
            else:
                ma_data[f'MA_{period}'] = 'N/A'

        current_price = hist['Close'].iloc[-1]
        ma_data['current_price'] = round(current_price, 2)

        return ma_data
    except Exception as e:
        return {'error': str(e)}

def calculate_volatility(stock_symbol, period_days=30):
    """Calculate historical volatility (annualized)"""
    try:
        ticker = yf.Ticker(stock_symbol)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=period_days + 10)

        hist = ticker.history(start=start_date, end=end_date)
        returns = hist['Close'].pct_change().dropna()

        volatility = returns.std() * np.sqrt(252) * 100  # Annualized volatility

        return {
            'volatility_percent': round(volatility, 2),
            'period_days': period_days,
            'avg_daily_return': round(returns.mean() * 100, 3)
        }
    except Exception as e:
        return {'error': str(e)}

# Test the financial tools
print("🧮 Testing Financial Calculation Tools...\n")

# Test risk/reward calculation
test_rr = calculate_risk_reward_ratio(entry_price=100, target_price=120, stop_loss=95)
print(f"Risk/Reward Example: {test_rr}:1")

# Test financial metrics
test_metrics = get_financial_metrics('AAPL')
print(f"\nAAPL Current Price: ${test_metrics.get('current_price', 'N/A')}")
print(f"AAPL P/E Ratio: {test_metrics.get('pe_ratio', 'N/A')}")

# Test moving averages
test_ma = calculate_moving_averages('AAPL')
print(f"\nAAPL Moving Averages:")
for key, value in test_ma.items():
    print(f"  {key}: {value}")

# Test volatility
test_vol = calculate_volatility('AAPL')
print(f"\nAAPL Volatility: {test_vol.get('volatility_percent', 'N/A')}%")

pretty_print("✅ Financial calculation tools ready!", "🧮 Tools Added", "blue")

🧮 Testing Financial Calculation Tools...

Risk/Reward Example: 4.0:1

AAPL Current Price: $272.41
AAPL P/E Ratio: 36.516087

AAPL Moving Averages:
  MA_20: 268.11
  MA_50: 255.7
  MA_200: 224.9
  current_price: 272.41

AAPL Volatility: 21.79%


In [13]:
# +++++ 🧪 LAB SECTION 3: Enhanced Task Definitions with Financial Tools
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Update tasks to incorporate financial metrics and risk analysis

def create_enhanced_investment_tasks(stock_symbol):
    """
    Enhanced tasks that use financial calculation tools
    """

    # Get financial metrics upfront
    metrics = get_financial_metrics(stock_symbol)
    ma_data = calculate_moving_averages(stock_symbol)
    vol_data = calculate_volatility(stock_symbol)

    # TASK 1: PORTFOLIO MANAGER COORDINATION
    coordination_task = Task(
        description=f"""As Portfolio Manager, coordinate comprehensive investment analysis for {stock_symbol}.

        DELEGATE these sub-tasks to your team:
        1. Market Analyst: Latest news and sentiment
        2. Research Analyst: Fundamental analysis with valuation metrics
        3. Risk-Focused Strategist: Trading strategy with risk management

        Financial Context:
        - Current Price: ${metrics.get('current_price', 'N/A')}
        - P/E Ratio: {metrics.get('pe_ratio', 'N/A')}
        - Volatility: {vol_data.get('volatility_percent', 'N/A')}%
        - 50-Day MA: ${ma_data.get('MA_50', 'N/A')}

        Create executive summary in EXACT format:

        🏢 {stock_symbol} INVESTMENT ANALYSIS
        💰 RECOMMENDATION: [BUY/SELL/HOLD]
        🎯 Target Price: $[X.XX]
        📉 Stop Loss: $[X.XX]
        ⏰ Timeframe: [Short/Medium/Long term]
        ⚖️ Risk/Reward: [X.XX:1]

        KEY INSIGHTS (3 bullet points max):
        • [Key point 1]
        • [Key point 2]
        • [Key point 3]

        ⚠️ RISKS: [Main risk in 1 sentence]""",
        agent=portfolio_manager,
        expected_output=f"Executive summary for {stock_symbol}"
    )

    # TASK 2: MARKET NEWS ANALYSIS
    market_task = Task(
        description=f"""Find latest news for {stock_symbol}:

        📰 MARKET NEWS ({stock_symbol})
        • [Headline 1]
        • [Headline 2]
        • [Headline 3]

        📊 SENTIMENT: [Positive/Negative/Neutral] - [Why]

        Current Price: ${metrics.get('current_price', 'N/A')}
        Keep concise - max 5 lines.""",
        agent=market_analyst,
        expected_output=f"News summary for {stock_symbol}"
    )

    # TASK 3: FUNDAMENTAL ANALYSIS WITH METRICS
    research_task = Task(
        description=f"""Analyze {stock_symbol} using these metrics:

        Given Data:
        - P/E Ratio: {metrics.get('pe_ratio', 'N/A')}
        - PEG Ratio: {metrics.get('peg_ratio', 'N/A')}
        - Price/Book: {metrics.get('price_to_book', 'N/A')}
        - Beta: {metrics.get('beta', 'N/A')}

        Provide analysis:

        📊 FUNDAMENTAL ANALYSIS ({stock_symbol})
        • Valuation: [Overvalued/Fair/Undervalued]
        • Growth: [Strong/Moderate/Weak]
        • Risk Level: [High/Medium/Low]

        💡 KEY INSIGHT: [Most important metric]

        Keep concise - max 5 lines.""",
        agent=research_analyst,
        expected_output=f"Fundamental analysis for {stock_symbol}"
    )

    # TASK 4: RISK-FOCUSED TRADING STRATEGY
    strategy_task = Task(
        description=f"""Create risk-managed trading strategy for {stock_symbol}:

        Current Data:
        - Price: ${metrics.get('current_price', 'N/A')}
        - Volatility: {vol_data.get('volatility_percent', 'N/A')}%
        - MA-50: ${ma_data.get('MA_50', 'N/A')}
        - MA-200: ${ma_data.get('MA_200', 'N/A')}

        💹 RISK-MANAGED STRATEGY ({stock_symbol})
        🎯 Action: [BUY/SELL/HOLD]
        💰 Entry: $[current price]
        🚀 Target: $[realistic target ABOVE entry if BUY]
        🛑 Stop Loss: $[5-8% below entry]
        ⚖️ Risk/Reward: [Calculate ratio, minimum 2:1]
        📊 Position Size: [Conservative/Moderate/Aggressive]

        📈 RATIONALE: [Why, including risk factors]

        CRITICAL:
        - If BUY: Target MUST be 10-15% above entry
        - Risk/Reward ratio must be minimum 2:1
        - Stop loss 5-8% below entry

        Keep concise - max 7 lines.""",
        agent=trading_strategist_v2,
        expected_output=f"Risk-managed strategy for {stock_symbol}"
    )

    return coordination_task, market_task, research_task, strategy_task

pretty_print("✅ Enhanced task system with financial metrics ready!", "📋 Tasks Updated", "blue")

In [16]:
# +++++ 🧪 LAB SECTION 4: Enhanced Analysis Function with Validation
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Run analysis with post-processing validation

def analyze_stock_enhanced(stock_symbol):
    """
    Enhanced analysis with financial metrics and validation
    """

    print("="*80)
    print(f"📊 ENHANCED INVESTMENT ANALYSIS: {stock_symbol}")
    print("="*80)

    # Display financial metrics first
    print(f"\n💰 FINANCIAL METRICS FOR {stock_symbol}:")
    print("-"*80)

    metrics = get_financial_metrics(stock_symbol)
    for key, value in metrics.items():
        if key != 'error':
            formatted_key = key.replace('_', ' ').title()
            if isinstance(value, float) and value > 1000000:
                value = f"${value/1e9:.2f}B"
            print(f"  {formatted_key}: {value}")

    ma_data = calculate_moving_averages(stock_symbol)
    print(f"\n📈 MOVING AVERAGES:")
    print("-"*80)
    for key, value in ma_data.items():
        if key != 'error':
            print(f"  {key}: ${value}" if value != 'N/A' else f"  {key}: N/A")

    vol_data = calculate_volatility(stock_symbol)
    print(f"\n📊 VOLATILITY ANALYSIS:")
    print("-"*80)
    print(f"  30-Day Volatility: {vol_data.get('volatility_percent', 'N/A')}%")
    print(f"  Avg Daily Return: {vol_data.get('avg_daily_return', 'N/A')}%")

    # Run CrewAI analysis
    print(f"\n🤖 RUNNING AI AGENT ANALYSIS...")
    print("-"*80 + "\n")

    coord_task, market_task, research_task, strategy_task = create_enhanced_investment_tasks(stock_symbol)

    investment_crew = Crew(
        agents=[portfolio_manager, market_analyst, research_analyst, trading_strategist_v2],
        tasks=[coord_task, market_task, research_task, strategy_task],
        verbose=True,  # Less verbose output
        max_iter=3
    )

    pretty_print(f"Analyzing {stock_symbol} with enhanced financial tools...", "💹 Processing", "yellow")

    try:
        result = investment_crew.kickoff()

        print("\n" + "="*80)
        print("✅ ANALYSIS COMPLETE")
        print("="*80)

        return result

    except Exception as e:
        pretty_print(f"⚠️ Analysis completed with warnings: {str(e)}", "⚠️ Warning", "yellow")
        return "Check output above for partial results"

pretty_print("✅ Enhanced analysis system ready!", "🎯 System Ready", "blue")

In [17]:
# +++++ 🧪 LAB SECTION 5: Test Multiple Stock Symbols
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Analyze multiple stocks and compare results

# List of stocks to analyze
test_stocks = ['AAPL', 'TSLA', 'GOOGL', 'NVDA']

# Store results for comparison
analysis_results = {}

print("\n" + "🚀 MULTI-STOCK ANALYSIS LAB")
print("="*80)
print(f"Testing stocks: {', '.join(test_stocks)}")
print("="*80 + "\n")

for stock in test_stocks:
    print(f"\n{'='*80}")
    print(f"🔍 ANALYZING: {stock}")
    print(f"{'='*80}\n")

    result = analyze_stock_enhanced(stock)
    analysis_results[stock] = result

    print(f"\n📊 RESULT FOR {stock}:")
    print("-"*80)
    print(result)
    print("\n" + "="*80 + "\n")

    # Add delay to avoid rate limiting
    import time
    if stock != test_stocks[-1]:  # Don't wait after last stock
        print("⏳ Waiting 30 seconds before next analysis to avoid rate limits...\n")
        time.sleep(30)

pretty_print(
    f"✅ Completed analysis of {len(test_stocks)} stocks!\n"
    f"Results stored in 'analysis_results' dictionary.",
    "🎊 Multi-Stock Analysis Complete",
    "blue"
)


🚀 MULTI-STOCK ANALYSIS LAB
Testing stocks: AAPL, TSLA, GOOGL, NVDA


🔍 ANALYZING: AAPL

📊 ENHANCED INVESTMENT ANALYSIS: AAPL

💰 FINANCIAL METRICS FOR AAPL:
--------------------------------------------------------------------------------
  Current Price: 272.41
  Pe Ratio: 36.516087
  Forward Pe: 32.780987
  Peg Ratio: N/A
  Price To Book: 54.580242
  Dividend Yield: 38.0
  Market Cap: 4033205501952
  52Week High: 277.32
  52Week Low: 169.21
  Beta: 1.109

📈 MOVING AVERAGES:
--------------------------------------------------------------------------------
  MA_20: $268.11
  MA_50: $255.7
  MA_200: $224.9
  current_price: $272.41

📊 VOLATILITY ANALYSIS:
--------------------------------------------------------------------------------
  30-Day Volatility: 21.79%
  Avg Daily Return: 0.228%

🤖 RUNNING AI AGENT ANALYSIS...
--------------------------------------------------------------------------------



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 33443fd2-9242-4701-bede-db02627bc940                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Portfolio Manager                                                                                │
│                                                                                                                 │
│  Task: As Portfolio Manager, coordinate comprehensive investment analysis for AAPL.                             │
│                                                                                                                 │
│          DELEGATE these sub-tasks to your team:                                                                 │
│          1. Market Analyst: Latest news and sentiment                                                           │
│          2. Research Analyst: Fundamental analysis with valuation metrics                                       │
│          3. Risk-Focused Strategist: Trading strategy with risk management                                      │
│                                                                                                                 │
│          Financial Context:                                                                                     │
│          - Current Price: $272.41                                                                               │
│          - P/E Ratio: 36.516087                                                                                 │
│          - Volatility: 21.79%                                                                                   │
│          - 50-Day MA: $255.7                                                                                    │
│                                                                                                                 │
│          Create executive summary in EXACT format:                                                              │
│                                                                                                                 │
│          🏢 AAPL INVESTMENT ANALYSIS                                                                            │
│          💰 RECOMMENDATION: [BUY/SELL/HOLD]                                                                     │
│          🎯 Target Price: $[X.XX]                                                                               │
│          📉 Stop Loss: $[X.XX]                                                                                  │
│          ⏰ Timeframe: [Short/Medium/Long term]                                                                 │
│          ⚖️ Risk/Reward: [X.XX:1]                                                                                │
│                                                                                                                 │
│          KEY INSIGHTS (3 bullet points max):                                                                    │
│          • [Key point 1]                                                                                        │
│          • [Key point 2]                                                                                        │
│          • [Key point 3]                                                                                        │
│                                                                                                                 │
│          ⚠️ RISKS: [Main risk in 1 sentence]                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

...

Repaired JSON: [{"task": "Gather the latest news and sentiment regarding GOOGL.", "context": "I need the most 
recent news articles, analyst opinions, and overall market sentiment related to GOOGL to assess the current 
investment climate.", "coworker": "Market News Analyst"}, {"task": "Conduct a fundamental analysis of GOOGL, 
focusing on valuation metrics such as P/E ratio, earnings growth, and other relevant financial indicators.", 
"context": "The current P/E ratio is 27.31. I need a detailed analysis of GOOGL's fundamentals, including 
comparisons with industry peers and historical performance.", "coworker": "Research Document Analyst"}, {"task": 
"Develop a trading strategy for GOOGL, including risk management techniques.", "context": "Given the current price 
of $276.41, volatility of 30.91%, and 50-day moving average of $257.24, I need a comprehensive trading strategy 
that includes entry and exit points, as well as risk management measures.", "coworker": "Risk-Focused Trading 
Strategist"}]

Repaired JSON: [{"task": "Gather the latest news and sentiment regarding GOOGL.", "context": "I need the most 
recent news articles, analyst opinions, and overall market sentiment related to GOOGL to assess the current 
investment climate.", "coworker": "Market News Analyst"}, {"task": "Conduct a fundamental analysis of GOOGL, 
focusing on valuation metrics such as P/E ratio, earnings growth, and other relevant financial indicators.", 
"context": "The current P/E ratio is 27.31. I need a detailed analysis of GOOGL's fundamentals, including 
comparisons with industry peers and historical performance.", "coworker": "Research Document Analyst"}, {"task": 
"Develop a trading strategy for GOOGL, including risk management techniques.", "context": "Given the current price 
of $276.41, volatility of 30.91%, and 50-day moving average of $257.24, I need a comprehensive trading strategy 
that includes entry and exit points, as well as risk management measures.", "coworker": "Risk-Focused Trading 
Strategist"}]

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Portfolio Manager                                                                                │
│                                                                                                                 │
│  Thought: Thought: I need to gather comprehensive information on GOOGL by delegating tasks to my team. I will   │
│  start by delegating the tasks to the respective coworkers.                                                     │
│                                                                                                                 │
│  Using Tool: Delegate work to coworker                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

KeyboardInterrupt: 

In [ ]:
# +++++ 🧪 LAB SECTION 6: Generate Comparison Summary
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Create a summary comparison of all analyzed stocks

print("\n" + "="*80)
print("📊 COMPARATIVE ANALYSIS SUMMARY")
print("="*80 + "\n")

comparison_data = []

for stock in test_stocks:
    print(f"\n🏢 {stock} SUMMARY:")
    print("-"*80)

    # Get financial metrics
    metrics = get_financial_metrics(stock)
    vol_data = calculate_volatility(stock)

    summary = {
        'Symbol': stock,
        'Current Price': f"${metrics.get('current_price', 'N/A')}",
        'P/E Ratio': metrics.get('pe_ratio', 'N/A'),
        'Beta': metrics.get('beta', 'N/A'),
        'Volatility': f"{vol_data.get('volatility_percent', 'N/A')}%",
        'AI Recommendation': analysis_results.get(stock, 'N/A')
    }

    for key, value in summary.items():
        print(f"  {key}: {value}")

    comparison_data.append(summary)

print("\n" + "="*80)
print("✅ ANALYSIS SUMMARY COMPLETE")
print("="*80)

pretty_print(
    "All stock analyses complete!\n\n"
    "Next steps for your report:\n"
    "1. Document what you changed (Risk-focused strategist)\n"
    "2. Note the financial tools added (P/E, MA, Volatility)\n"
    "3. Compare recommendations across stocks\n"
    "4. Discuss observations (risk levels, valuations, strategies)\n"
    "5. Export findings to a 1-2 page PDF",
    "📝 Report Guidelines",
    "blue"
)

In [ ]:
# +++++ 🧪 LAB SECTION 7: Key Observations Template
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Generate a template for your lab report

report_template = f"""
# 🧪 CREWAI INVESTMENT ANALYSIS LAB REPORT

## 1. MODIFICATIONS MADE

### Agent Modifications:
- **Trading Strategist** → **Risk-Focused Trading Strategist**
  - Enhanced with capital preservation focus
  - Minimum 2:1 risk/reward requirements
  - Volatility-aware position sizing
  - Conservative stop-loss strategies

### Tools Added:
- **calculate_risk_reward_ratio()**: Computes R/R ratios for trade validation
- **get_financial_metrics()**: Fetches P/E, PEG, Price/Book, Beta using yfinance
- **calculate_moving_averages()**: Computes 20, 50, 200-day MAs for trend analysis
- **calculate_volatility()**: Measures 30-day historical volatility (annualized)

---

## 2. STOCKS TESTED

Analyzed the following symbols:
{chr(10).join([f'- **{stock}**: {get_financial_metrics(stock).get("current_price", "N/A")}' for stock in test_stocks])}

---

## 3. OBSERVATIONS

### Stock-Specific Findings:

"""

for stock in test_stocks:
    metrics = get_financial_metrics(stock)
    vol = calculate_volatility(stock)
    report_template += f"""
**{stock}:**
- Current Price: ${metrics.get('current_price', 'N/A')}
- P/E Ratio: {metrics.get('pe_ratio', 'N/A')}
- Volatility: {vol.get('volatility_percent', 'N/A')}%
- Beta: {metrics.get('beta', 'N/A')}
- AI Recommendation: [See detailed output above]

"""

report_template += """
### Comparative Insights:

1. **Valuation Differences**: [Discuss P/E ratios across stocks]
2. **Risk Profiles**: [Compare volatility and beta values]
3. **AI Recommendations**: [Note differences in BUY/SELL/HOLD calls]
4. **Tool Effectiveness**: [How financial metrics improved analysis]

### Key Learnings:

- The risk-focused strategist provided more conservative recommendations
- Financial metrics helped contextualize AI suggestions
- Volatility analysis influenced position sizing recommendations
- Moving averages provided trend confirmation

---

## 4. CONCLUSION

[Summarize: What worked well, what could be improved, how financial tools enhanced the analysis]

---

**Date:** {datetime.now().strftime('%B %d, %Y')}
**Total Execution Time:** [Record from your notebook]
**Tools Used:** CrewAI, yfinance, OpenAI GPT-4o-mini
"""

# Print template
print(report_template)

# Optionally save to file
with open('lab_report_template.txt', 'w') as f:
    f.write(report_template)

pretty_print(
    "✅ Report template generated!\n"
    "Saved to: lab_report_template.txt\n\n"
    "Next: Copy this template, fill in your observations, and export as PDF",
    "📄 Template Ready",
    "blue"
)

In [ ]:
# +++++ 🧪 LAB SECTION 8: Submission Checklist
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Final verification before submission

print("\n" + "="*80)
print("✅ LAB SUBMISSION CHECKLIST")
print("="*80 + "\n")

checklist = {
    "1. Modified Strategic Agent": "✅ Changed to Risk-Focused Trading Strategist",
    "2. Added Financial Tools": "✅ 4 new calculation functions (R/R, P/E, MA, Volatility)",
    "3. Tested Multiple Stocks": f"✅ Analyzed {len(test_stocks)} stocks: {', '.join(test_stocks)}",
    "4. Generated Comparison": "✅ Comparative analysis completed",
    "5. Created Report Template": "✅ Report template saved to file",
}

for item, status in checklist.items():
    print(f"{status}")
    print(f"   {item}")
    print()

print("="*80)
print("📝 NEXT STEPS FOR PDF SUBMISSION:")
print("="*80)
print("""
1. Review the report template above
2. Add your personal observations and insights
3. Include 1-2 key charts/visualizations if desired
4. Format as a clean 1-2 page PDF document
5. Ensure all sections are complete:
   - What you changed
   - What symbols you tested
   - What you observed
6. Submit your PDF through the course portal

""")

pretty_print(
    "🎉 Congratulations! You've completed the CrewAI Advanced Lab!\n\n"
    "You successfully:\n"
    "• Modified agent behavior with risk management focus\n"
    "• Integrated real financial calculation tools\n"
    "• Tested multiple stock symbols\n"
    "• Generated comprehensive comparative analysis\n\n"
    "Your PDF report should highlight these achievements!",
    "🏆 Lab Complete",
    "blue"
)

# Record completion time
end_time = time.time()
total_time = end_time - start_time
print(f"\n⏱️ Total Lab Execution Time: {total_time/60:.2f} minutes")

In [ ]:
# +++++ 🎓 Lab Completion Certificate (Dropbox Version)
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Download and run completion certificate from Dropbox

import requests

print("🔄 Loading completion certificate from Dropbox...")

# Download and execute the completion script
dropbox_url = "https://www.dropbox.com/scl/fi/5molmat6myeqaf96kp50v/CrewAI_Completiton.py?rlkey=7v7yaf9gi5hupkxiqaits50rd&dl=1"

response = requests.get(dropbox_url)
exec(response.text)